In [1]:
import csv
from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem

# =========================
# QUERY MOLECULE
# =========================
query_smiles = "CCOC"
query_mol = Chem.MolFromSmiles(query_smiles)

if not query_mol:
    raise ValueError("Invalid query SMILES")

query_fp = AllChem.GetMorganFingerprintAsBitVect(query_mol, radius=2, nBits=2048)

# =========================
# LIBRARY (replace with SDF if needed)
# =========================
smiles_db = [
    "CCO",
    "CCN",
    "CCCO",
    "CCOC",
    "c1ccccc1",
    "CC(=O)O",
    "CCS",
]

# =========================
# SCREENING
# =========================
results = []

for smi in smiles_db:
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        continue

    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=2048)

    sim = DataStructs.TanimotoSimilarity(query_fp, fp)

    results.append((smi, sim))

# =========================
# SORT BY SIMILARITY
# =========================
results.sort(key=lambda x: x[1], reverse=True)

# =========================
# SAVE TO CSV
# =========================
output_file = "tanimoto_results.csv"

with open(output_file, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["SMILES", "TanimotoSimilarity"])
    writer.writerows(results)

print(f"\nSaved results to {output_file}\n")

# =========================
# PRINT TOP HITS
# =========================
print("Top similar molecules:\n")

for smi, score in results:
    print(f"{smi:15}  Tanimoto = {score:.3f}")


Saved results to tanimoto_results.csv

Top similar molecules:

CCOC             Tanimoto = 1.000
CCO              Tanimoto = 0.273
CCN              Tanimoto = 0.273
CCS              Tanimoto = 0.273
CCCO             Tanimoto = 0.231
CC(=O)O          Tanimoto = 0.071
c1ccccc1         Tanimoto = 0.000


In [2]:
# Scalable similarity screening (SMILES or SDF database)

In [8]:
import csv
import os
from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem

# =========================
# SETTINGS
# =========================
TOP_N = 50
THRESHOLD = 0.2
RADIUS = 2
N_BITS = 2048

# =========================
# USER INPUT
# =========================
query_path = input("Enter query molecule path (.smi/.txt/.sdf or SMILES): ").strip()
db_path = input("Enter database path (.smi/.txt/.sdf): ").strip()

# =========================
# LOAD QUERY
# =========================
def load_query(q):
    ext = os.path.splitext(q)[1].lower()

    if ext == ".sdf":
        mol = Chem.SDMolSupplier(q)[0]
        return mol

    elif ext in [".smi", ".txt"]:
        with open(q) as f:
            smi = f.readline().split()[0]
        return Chem.MolFromSmiles(smi)

    else:
        return Chem.MolFromSmiles(q)

query_mol = load_query(query_path)

if query_mol is None:
    raise ValueError("Invalid query molecule")

query_fp = AllChem.GetMorganFingerprintAsBitVect(query_mol, RADIUS, nBits=N_BITS)

# =========================
# DATABASE LOADERS
# =========================
def load_smiles_db(file_path):
    with open(file_path) as f:
        for i, line in enumerate(f):
            smi = line.split()[0]
            mol = Chem.MolFromSmiles(smi)
            if mol:
                yield f"mol_{i}", smi, mol

def load_sdf_db(file_path):
    supplier = Chem.SDMolSupplier(file_path, sanitize=True, removeHs=False)

    for i, mol in enumerate(supplier):
        if mol:
            # use existing name if present
            mol_id = mol.GetProp("_Name") if mol.HasProp("_Name") else f"mol_{i}"
            smi = Chem.MolToSmiles(mol)
            yield mol_id, smi, mol

ext = os.path.splitext(db_path)[1].lower()

if ext == ".sdf":
    db_generator = load_sdf_db(db_path)
elif ext in [".smi", ".txt"]:
    db_generator = load_smiles_db(db_path)
else:
    raise ValueError("Unsupported DB format")

# =========================
# SCREENING
# =========================
results = []

for mol_id, smi, mol in db_generator:
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, RADIUS, nBits=N_BITS)
    sim = DataStructs.TanimotoSimilarity(query_fp, fp)

    if sim >= THRESHOLD:
        results.append((mol_id, smi, sim, mol))

# =========================
# SORT & SELECT
# =========================
results.sort(key=lambda x: x[2], reverse=True)
top_hits = results[:TOP_N]

print(f"\nTotal hits above threshold: {len(results)}")
print(f"Showing top {len(top_hits)} results\n")

# =========================
# SAVE CSV
# =========================
csv_file = "tanimoto_results.csv"

with open(csv_file, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["Mol_ID", "SMILES", "TanimotoScore"])

    for mol_id, smi, score, _ in top_hits:
        writer.writerow([mol_id, smi, score])

print(f"Saved CSV: {csv_file}")

# =========================
# SAVE SDF WITH SCORES
# =========================
sdf_file = "tanimoto_hits.sdf"
writer = Chem.SDWriter(sdf_file)

for mol_id, smi, score, mol in top_hits:
    mol.SetProp("_Name", mol_id)
    mol.SetProp("SMILES", smi)
    mol.SetProp("TanimotoScore", str(score))
    writer.write(mol)

writer.close()

print(f"Saved SDF: {sdf_file}")

# =========================
# PRINT RESULTS
# =========================
for mol_id, smi, score, _ in top_hits:
    print(f"{mol_id:10}  {smi:20}  Score = {score:.3f}")

Enter query molecule path (.smi/.txt/.sdf or SMILES): CNC(=O)C1=NN=C(C=C1NC2=CC=CC(=C2OC)C3=NN(C=N3)C)NC(=O)C4CC4
Enter database path (.smi/.txt/.sdf): C:/Users/ravit/Desktop/TYK2/TESTTYK2minus.sdf

Total hits above threshold: 254
Showing top 50 results

Saved CSV: tanimoto_results.csv
Saved SDF: tanimoto_hits.sdf
mol_212_taut_0  [H]c1c([H])c(-c2nc([H])n(C([H])([H])[H])n2)c(OC([H])([H])[H])c(N([H])c2c(C(=O)N([H])C([2H])([2H])[2H])nnc(N([H])C(=O)C3([H])C([H])([H])C3([H])[H])c2[H])c1[H]  Score = 0.313
mol_385_taut_0  [H]c1c([H])c(-c2nc([H])n(C([H])([H])[H])n2)c(OC([H])([H])[H])c(N([H])c2c(C(=O)N([H])C([2H])([2H])[2H])nnc(N([H])C(=O)C3([H])C([H])([H])C3([H])[H])c2[H])c1[H]  Score = 0.313
mol_491_taut_0  [H]c1c([H])c(-c2nc([H])n(C([H])([H])[H])n2)c(OC([H])([H])[H])c(N([H])c2c(C(=O)N([H])C([2H])([2H])[2H])nnc(N([H])C(=O)C3([H])C([H])([H])C3([H])[H])c2[H])c1[H]  Score = 0.313
mol_535_taut_0  [H]c1c([H])c(-c2nc([H])n(C([H])([H])[H])n2)c(OC([H])([H])[H])c(N([H])c2c(C(=O)N([H])C([2H])([2H])[2H]

In [9]:
# Pharmacophore screening

In [11]:
import os
import csv
from rdkit import Chem
from rdkit.Chem import ChemicalFeatures
from rdkit import RDConfig

# =========================
# SETTINGS
# =========================
TOP_N = 50
THRESHOLD = 2   # minimum feature matches

# =========================
# USER INPUT
# =========================
query_path = input("Enter query molecule (.sdf/.smi or SMILES): ").strip()
db_path = input("Enter database (.sdf/.smi/.txt): ").strip()

# =========================
# LOAD FEATURE FACTORY
# =========================
fdefName = os.path.join(RDConfig.RDDataDir, 'BaseFeatures.fdef')
feat_factory = ChemicalFeatures.BuildFeatureFactory(fdefName)

# =========================
# LOAD QUERY
# =========================
def load_query(q):
    ext = os.path.splitext(q)[1].lower()

    if ext == ".sdf":
        return Chem.SDMolSupplier(q)[0]

    elif ext in [".smi", ".txt"]:
        with open(q) as f:
            smi = f.readline().split()[0]
        return Chem.MolFromSmiles(smi)

    else:
        return Chem.MolFromSmiles(q)

query_mol = load_query(query_path)

if query_mol is None:
    raise ValueError("Invalid query molecule")

query_feats = feat_factory.GetFeaturesForMol(query_mol)

# =========================
# DATABASE LOADERS
# =========================
def load_smiles_db(path):
    with open(path) as f:
        for i, line in enumerate(f):
            smi = line.split()[0]
            mol = Chem.MolFromSmiles(smi)
            if mol:
                yield f"mol_{i}", smi, mol

def load_sdf_db(path):
    supplier = Chem.SDMolSupplier(path, removeHs=False)
    for i, mol in enumerate(supplier):
        if mol:
            mol_id = mol.GetProp("_Name") if mol.HasProp("_Name") else f"mol_{i}"
            smi = Chem.MolToSmiles(mol)
            yield mol_id, smi, mol

ext = os.path.splitext(db_path)[1].lower()

if ext == ".sdf":
    db_generator = load_sdf_db(db_path)
elif ext in [".smi", ".txt"]:
    db_generator = load_smiles_db(db_path)
else:
    raise ValueError("Unsupported DB format")

# =========================
# PHARMACOPHORE SCORING
# =========================
def score_pharmacophore(query_feats, mol_feats):
    q_types = [f.GetFamily() for f in query_feats]
    m_types = [f.GetFamily() for f in mol_feats]

    # simple overlap count
    score = 0
    for ft in q_types:
        if ft in m_types:
            score += 1

    return score

# =========================
# SCREENING
# =========================
results = []

for mol_id, smi, mol in db_generator:
    feats = feat_factory.GetFeaturesForMol(mol)

    score = score_pharmacophore(query_feats, feats)

    if score >= THRESHOLD:
        results.append((mol_id, smi, score, mol))

# =========================
# SORT
# =========================
results.sort(key=lambda x: x[2], reverse=True)
top_hits = results[:TOP_N]

print(f"\nTotal hits above threshold: {len(results)}")
print(f"Top {len(top_hits)} hits\n")

# =========================
# SAVE CSV
# =========================
with open("C:/Users/ravit/Desktop/TYK2/pharmacophore_hits.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["Mol_ID", "SMILES", "PharmacophoreScore"])

    for mol_id, smi, score, _ in top_hits:
        writer.writerow([mol_id, smi, score])

# =========================
# SAVE SDF
# =========================
writer = Chem.SDWriter("pharmacophore_hits.sdf")

for mol_id, smi, score, mol in top_hits:
    mol.SetProp("_Name", mol_id)
    mol.SetProp("SMILES", smi)
    mol.SetProp("PharmacophoreScore", str(score))
    writer.write(mol)

writer.close()

print("Saved pharmacophore results\n")

# =========================
# PRINT
# =========================
for mol_id, smi, score, _ in top_hits:
    print(f"{mol_id:10}  {smi:20}  Score = {score}")

Enter query molecule (.sdf/.smi or SMILES): CNC(=O)C1=NN=C(C=C1NC2=CC=CC(=C2OC)C3=NN(C=N3)C)NC(=O)C4CC4
Enter database (.sdf/.smi/.txt): C:/Users/ravit/Desktop/TYK2/TESTTYK2minus.sdf

Total hits above threshold: 5097
Top 50 hits

Saved pharmacophore results

mol_0_taut_0  [H]c1nc(N([H])[H])c2c([H])c(-c3c([H])c([H])c(S(=O)(=O)N([H])C(C([H])([H])[H])(C([H])([H])[H])C([H])([H])[H])c([H])c3[H])sc2c1C(=O)N([H])C([H])([H])C([H])([H])C([H])([H])N1C([H])([H])C([H])([H])OC([H])([H])C1([H])[H]  Score = 17
mol_0_taut_1  [H]N=c1c2c([H])c(-c3c([H])c([H])c(S(=O)(=O)N([H])C(C([H])([H])[H])(C([H])([H])[H])C([H])([H])[H])c([H])c3[H])sc2c(C(=O)N([H])C([H])([H])C([H])([H])C([H])([H])N2C([H])([H])C([H])([H])OC([H])([H])C2([H])[H])c([H])n1[H]  Score = 17
mol_0_taut_2  [H]N=c1c2c([H])c(-c3c([H])c([H])c(S(=O)(=O)N([H])C(C([H])([H])[H])(C([H])([H])[H])C([H])([H])[H])c([H])c3[H])sc2c(C(=NC([H])([H])C([H])([H])C([H])([H])N2C([H])([H])C([H])([H])OC([H])([H])C2([H])[H])O[H])c([H])n1[H]  Score = 17
mol_1_taut_0  [

In [12]:
#Pharmacophore screening with Conformations 

In [14]:
import os
import csv
import itertools
from rdkit import Chem
from rdkit.Chem import AllChem, ChemicalFeatures
from rdkit import RDConfig

# =========================
# SETTINGS
# =========================
TOP_N = 20
DIST_TOL = 1.5   # Å tolerance
MAX_CONFS = 5

# =========================
# INPUT
# =========================
query_path = input("Enter query molecule (.sdf/.smi or SMILES): ").strip()
db_path = input("Enter database (.sdf/.smi/.txt): ").strip()

# =========================
# FEATURE FACTORY
# =========================
fdefName = os.path.join(RDConfig.RDDataDir, 'BaseFeatures.fdef')
feat_factory = ChemicalFeatures.BuildFeatureFactory(fdefName)

# =========================
# LOAD MOLECULE
# =========================
def load_mol(inp):
    ext = os.path.splitext(inp)[1].lower()

    if ext == ".sdf":
        return Chem.SDMolSupplier(inp)[0]
    elif ext in [".smi", ".txt"]:
        with open(inp) as f:
            smi = f.readline().split()[0]
        return Chem.MolFromSmiles(smi)
    else:
        return Chem.MolFromSmiles(inp)

# =========================
# GENERATE CONFORMERS
# =========================
def generate_conformers(mol):
    mol = Chem.AddHs(mol)
    ids = AllChem.EmbedMultipleConfs(mol, numConfs=MAX_CONFS)
    for cid in ids:
        AllChem.UFFOptimizeMolecule(mol, confId=cid)
    return mol

# =========================
# GET FEATURES + COORDS
# =========================
def get_features(mol, confId=0):
    feats = feat_factory.GetFeaturesForMol(mol)
    result = []

    conf = mol.GetConformer(confId)

    for f in feats:
        pos = f.GetPos()
        result.append((f.GetFamily(), pos))

    return result

# =========================
# DISTANCE MATRIX
# =========================
def pairwise_dist(feats):
    pairs = []
    for (f1, p1), (f2, p2) in itertools.combinations(feats, 2):
        d = (p1 - p2).Length()
        pairs.append((f1, f2, d))
    return pairs

# =========================
# MATCH SCORING
# =========================
def score_match(query_pairs, target_pairs):
    score = 0

    for qf1, qf2, qd in query_pairs:
        for tf1, tf2, td in target_pairs:

            # feature type match (unordered)
            if {qf1, qf2} == {tf1, tf2}:
                if abs(qd - td) <= DIST_TOL:
                    score += 1

    return score

# =========================
# LOAD QUERY
# =========================
query_mol = load_mol(query_path)
query_mol = generate_conformers(query_mol)

query_feats = get_features(query_mol, confId=0)
query_pairs = pairwise_dist(query_feats)

# =========================
# DATABASE LOADER
# =========================
def load_db(path):
    ext = os.path.splitext(path)[1].lower()

    if ext == ".sdf":
        supplier = Chem.SDMolSupplier(path)
        for i, mol in enumerate(supplier):
            if mol:
                mol_id = mol.GetProp("_Name") if mol.HasProp("_Name") else f"mol_{i}"
                yield mol_id, mol

    else:
        with open(path) as f:
            for i, line in enumerate(f):
                smi = line.split()[0]
                mol = Chem.MolFromSmiles(smi)
                if mol:
                    yield f"mol_{i}", mol

# =========================
# SCREENING
# =========================
results = []

for mol_id, mol in load_db(db_path):

    try:
        mol = generate_conformers(mol)

        best_score = 0

        for cid in range(mol.GetNumConformers()):
            feats = get_features(mol, confId=cid)
            pairs = pairwise_dist(feats)

            score = score_match(query_pairs, pairs)

            if score > best_score:
                best_score = score

        if best_score > 0:
            smi = Chem.MolToSmiles(mol)
            results.append((mol_id, smi, best_score, mol))

    except:
        continue

# =========================
# SORT
# =========================
results.sort(key=lambda x: x[2], reverse=True)
top_hits = results[:TOP_N]

# =========================
# SAVE CSV
# =========================
with open("C:/Users/ravit/Desktop/TYK2/pharma3d_hits.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["Mol_ID", "SMILES", "Pharma3DScore"])

    for mol_id, smi, score, _ in top_hits:
        writer.writerow([mol_id, smi, score])

# =========================
# SAVE SDF
# =========================
writer = Chem.SDWriter("pharma3d_hits.sdf")

for mol_id, smi, score, mol in top_hits:
    mol.SetProp("_Name", mol_id)
    mol.SetProp("SMILES", smi)
    mol.SetProp("Pharma3DScore", str(score))
    writer.write(mol)

writer.close()

print("\nTop hits:\n")
for mol_id, smi, score, _ in top_hits:
    print(f"{mol_id:10}  {smi:20}  Score = {score}")

Enter query molecule (.sdf/.smi or SMILES): CNC(=O)C1=NN=C(C=C1NC2=CC=CC(=C2OC)C3=NN(C=N3)C)NC(=O)C4CC4
Enter database (.sdf/.smi/.txt): C:/Users/ravit/Desktop/TYK2/TESTTYK2minus.sdf

Top hits:

mol_166_taut_1  [H]OC1=NC([H])([H])c2c1c1c3c([H])c([H])c([H])c([H])c3n3c1c1c2c2c([H])c([H])c([H])c([H])c2n1[C@@]1(C([H])([H])[H])O[C@]3([H])C([H])([H])[C@@]([H])(N(C(=O)c2c([H])c([H])c([H])c([H])c2[H])C([H])([H])[H])[C@@]1([H])OC([H])([H])[H]  Score = 5405
mol_1042_taut_2  [H]OC1=NC([H])([H])c2c1c1c3c([H])c([H])c([H])c([H])c3n3c1c1c2c2c([H])c([H])c([H])c([H])c2n1[C@@]1(C([H])([H])[H])O[C@]3([H])C([H])([H])[C@@]([H])(N([H])C([H])([H])C([H])([H])OC([H])([H])C([H])([H])OC([H])([H])C([H])([H])OC([H])([H])C([H])([H])N=C(O[H])C([H])([H])C([H])([H])C2=[N+]3C(=C([H])c4c(C([H])([H])[H])c([H])c(C([H])([H])[H])n4[B-]3(F)F)C([H])=C2[H])[C@@]1([H])OC([H])([H])[H]  Score = 4921
mol_1042_taut_1  [H]OC(=NC([H])([H])C([H])([H])OC([H])([H])C([H])([H])OC([H])([H])C([H])([H])OC([H])([H])C([H])([H])N([H])[C@]1([H])

In [ ]:
# shape screening

In [1]:
from rdkit import Chem
from rdkit.Chem import AllChem, rdShapeHelpers
import csv

# -------- User Inputs --------
query_input = input("Enter query SMILES or path to query SDF: ").strip()
db_input = input("Enter database file path (.smi or .sdf): ").strip()
output_csv = input("Enter output CSV filename (e.g., results.csv): ").strip()


# -------- Load Query --------
def load_query(q):
    if q.endswith(".sdf"):
        suppl = Chem.SDMolSupplier(q)
        mol = next((m for m in suppl if m is not None), None)
    else:
        mol = Chem.MolFromSmiles(q)

    if mol is None:
        raise ValueError("Invalid query input!")

    mol = Chem.AddHs(mol)
    AllChem.EmbedMolecule(mol, randomSeed=42)
    AllChem.UFFOptimizeMolecule(mol)
    return mol


# -------- Load Database --------
def load_database(path):
    mols = []

    if path.endswith(".sdf"):
        suppl = Chem.SDMolSupplier(path)
        for i, mol in enumerate(suppl):
            if mol:
                smi = Chem.MolToSmiles(mol)
                mol_id = mol.GetProp("_Name") if mol.HasProp("_Name") else f"mol_{i}"
                mols.append((mol_id, smi, mol))
    else:
        with open(path) as f:
            for i, line in enumerate(f):
                parts = line.strip().split()
                if len(parts) >= 1:
                    smi = parts[0]
                    mol_id = parts[1] if len(parts) > 1 else f"mol_{i}"
                    mol = Chem.MolFromSmiles(smi)
                    if mol:
                        mols.append((mol_id, smi, mol))

    return mols


# -------- Prepare Molecule --------
def prepare_mol(m):
    m = Chem.AddHs(m)
    try:
        AllChem.EmbedMolecule(m, randomSeed=42)
        AllChem.UFFOptimizeMolecule(m)
        return m
    except:
        return None


# -------- Shape Screening --------
def run_screening(query, db):
    results = []

    for mol_id, smi, mol in db:
        mol3d = prepare_mol(mol)
        if mol3d is None:
            continue

        try:
            score = 1 - rdShapeHelpers.ShapeTanimotoDist(query, mol3d)
            results.append((mol_id, smi, score))
        except:
            continue

    results.sort(key=lambda x: x[2], reverse=True)
    return results


# -------- Run --------
print("Loading query...")
query_mol = load_query(query_input)

print("Loading database...")
db_mols = load_database(db_input)

print(f"Screening {len(db_mols)} molecules...")
results = run_screening(query_mol, db_mols)

# -------- Save CSV --------
with open(output_csv, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["ID", "SMILES", "ShapeScore"])
    writer.writerows(results)

print(f"Done! Results saved to {output_csv}")

Enter query SMILES or path to query SDF: CNC(=O)C1=NN=C(C=C1NC2=CC=CC(=C2OC)C3=NN(C=N3)C)NC(=O)C4CC4
Enter database file path (.smi or .sdf): C:/Users/ravit/Desktop/TYK2/TESTTYK2minus.sdf
Enter output CSV filename (e.g., results.csv): C:/Users/ravit/Desktop/TYK2/shaperesult.csv
Loading query...
Loading database...
Screening 5097 molecules...
Done! Results saved to C:/Users/ravit/Desktop/TYK2/shaperesult.csv


In [2]:
# # shape screening with conformations

In [3]:
from rdkit import Chem
from rdkit.Chem import AllChem, rdShapeHelpers
import csv

# -------- User Inputs --------
query_input = input("Enter query SMILES or path to query SDF: ").strip()
db_input = input("Enter database file path (.smi or .sdf): ").strip()
output_csv = input("Enter output CSV filename (e.g., results.csv): ").strip()
num_confs = int(input("Number of conformers per molecule (e.g., 10–50): ").strip() or 20)


# -------- Conformer Generation --------
def gen_conformers(mol, nconf=20):
    mol = Chem.AddHs(mol)
    params = AllChem.ETKDGv3()
    params.randomSeed = 42
    params.numThreads = 0  # use all cores

    try:
        conf_ids = AllChem.EmbedMultipleConfs(mol, numConfs=nconf, params=params)
        for cid in conf_ids:
            try:
                AllChem.UFFOptimizeMolecule(mol, confId=cid)
            except:
                pass
        return mol, list(conf_ids)
    except:
        return None, []


# -------- Load Query --------
def load_query(q):
    if q.endswith(".sdf"):
        suppl = Chem.SDMolSupplier(q)
        mol = next((m for m in suppl if m is not None), None)
    else:
        mol = Chem.MolFromSmiles(q)

    if mol is None:
        raise ValueError("Invalid query input!")

    mol3d, confs = gen_conformers(mol, num_confs)
    if mol3d is None or not confs:
        raise ValueError("Failed to generate conformers for query!")

    return mol3d, confs


# -------- Load Database --------
def load_database(path):
    mols = []

    if path.endswith(".sdf"):
        suppl = Chem.SDMolSupplier(path)
        for i, mol in enumerate(suppl):
            if mol:
                smi = Chem.MolToSmiles(mol)
                mol_id = mol.GetProp("_Name") if mol.HasProp("_Name") else f"mol_{i}"
                mols.append((mol_id, smi, mol))
    else:
        with open(path) as f:
            for i, line in enumerate(f):
                parts = line.strip().split()
                if len(parts) >= 1:
                    smi = parts[0]
                    mol_id = parts[1] if len(parts) > 1 else f"mol_{i}"
                    mol = Chem.MolFromSmiles(smi)
                    if mol:
                        mols.append((mol_id, smi, mol))

    return mols


# -------- Shape Screening (Multi-Conformer) --------
def multi_conf_shape_screen(query_mol, query_confs, db_mols):
    results = []

    for mol_id, smi, mol in db_mols:
        mol3d, db_confs = gen_conformers(mol, num_confs)
        if mol3d is None or not db_confs:
            continue

        best_score = 0.0

        for qc in query_confs:
            for dc in db_confs:
                try:
                    score = 1 - rdShapeHelpers.ShapeTanimotoDist(
                        query_mol, mol3d, confId1=qc, confId2=dc
                    )
                    if score > best_score:
                        best_score = score
                except:
                    continue

        results.append((mol_id, smi, best_score))

    results.sort(key=lambda x: x[2], reverse=True)
    return results


# -------- Run --------
print("Preparing query...")
query_mol, query_confs = load_query(query_input)

print("Loading database...")
db_mols = load_database(db_input)

print(f"Screening {len(db_mols)} molecules with {num_confs} conformers each...")
results = multi_conf_shape_screen(query_mol, query_confs, db_mols)

# -------- Save CSV --------
with open(output_csv, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["ID", "SMILES", "ShapeScore"])
    writer.writerows(results)

print(f"Done! Results saved to {output_csv}")

Enter query SMILES or path to query SDF: CNC(=O)C1=NN=C(C=C1NC2=CC=CC(=C2OC)C3=NN(C=N3)C)NC(=O)C4CC4
Enter database file path (.smi or .sdf): C:/Users/ravit/Desktop/TYK2/TESTTYK2minus.sdf
Enter output CSV filename (e.g., results.csv): C:/Users/ravit/Desktop/TYK2/tyk2shapconf.csv
Number of conformers per molecule (e.g., 10–50): 10
Preparing query...
Loading database...
Screening 5097 molecules with 10 conformers each...
Done! Results saved to C:/Users/ravit/Desktop/TYK2/tyk2shapconf.csv
